In [ ]:
# EPOCHS_DIR = '/kaggle/input/datasets/nguyenkienn/v1-dataset/v1'
# RESULTS_DIR = '/kaggle/working/'
# FOLDS_JSON_PATH = '/kaggle/input/datasets/nguyenkienn/folds-data/folds.json'
# OUTPUT_CSV = '/kaggle/working/final_predictions_cross_subject.csv'

from src.eegoddball.config import EPOCHS_DIR, RESULTS_DIR, FOLDS_PATH as FOLDS_JSON_PATH

In [2]:
import os
import os
import mne
import numpy as np
import pandas as pd
import gc
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import json
import hashlib

from scipy.signal import welch
from scipy.integrate import simpson
import pywt
from mne.preprocessing import Xdawn
from mne import compute_covariance

In [ ]:
OUTPUT_CSV = os.path.join(RESULTS_DIR, 'final_predictions.csv')

In [4]:
def load_single_subject(sub_id, epochs_dir):
    fif_path = os.path.join(epochs_dir, f"sub-{sub_id}_task-oddball-epo.fif")
    if not os.path.exists(fif_path):
        return None, None, None
        
    epochs = mne.read_epochs(fif_path, preload=True, verbose=False)[["standard", "target"]]
    X = epochs.get_data().astype(np.float32)
    y = (epochs.events[:, -1] == 2).astype(np.int8)
    trial_id = epochs.metadata["trial_id"].to_numpy()
    
    del epochs
    return X, y, trial_id
def extract_features_in_chunks(X, xdawn_model, chunk_size=1500):
    features_list = []
    n_samples = len(X)
    
    for i in range(0, n_samples, chunk_size):
        X_chunk = X[i:i+chunk_size]
        
        # 1. Apply xDAWN cho lô nhỏ
        X_chunk_xdawn = apply_xdawn(xdawn_model, X_chunk)
        
        # 2. Trích xuất đặc trưng (Wavelet, Bandpower...)
        chunk_feat = combine_features(X_chunk, X_chunk_xdawn).astype(np.float32)
        features_list.append(chunk_feat)
        
        # DỌN RÁC NGAY SAU MỖI LÔ
        del X_chunk, X_chunk_xdawn, chunk_feat
        gc.collect()
        
    # Gộp các lô đặc trưng lại thành ma trận 2D cuối cùng
    return np.concatenate(features_list, axis=0)


In [5]:
def fit_xdawn(X_train, y_train, sfreq=250, n_components=2, max_trials=3000):
    """
    Hàm huấn luyện xDAWN đã tối ưu chống tràn RAM và chống lỗi Ma Trận Singluar (LinAlgError).
    """
    import gc
    import mne
    import numpy as np
    
    n_trials = X_train.shape[0]
    
    # CHIẾN THUẬT 1: Giới hạn số trials để chống tràn RAM
    if n_trials > max_trials:
        np.random.seed(42)
        indices = np.random.choice(n_trials, max_trials, replace=False)
        X_sub = X_train[indices].copy() # Copy để không đụng tới data gốc
        y_sub = y_train[indices]
    else:
        X_sub = X_train.copy()
        y_sub = y_train

    # CHIẾN THUẬT 2: Khử DC offset (Thay thế cho Baseline correction)
    # Trừ đi giá trị trung bình theo thời gian của từng kênh, từng trial
    X_sub = X_sub - np.mean(X_sub, axis=-1, keepdims=True)

    n_channels = X_sub.shape[1]
    ch_names = [f'EEG{i}' for i in range(n_channels)]
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    
    events = np.column_stack([
        np.arange(len(y_sub)), 
        np.zeros(len(y_sub), dtype=int), 
        y_sub
    ])
    
    epochs_tr = mne.EpochsArray(X_sub.astype(np.float64), info, events=events, verbose=False)
    
    # CHIẾN THUẬT 3: Dùng 'ledoit_wolf' có regularization giúp triệt tiêu LinAlgError
    # Vì max_trials chỉ còn 3000 nên chạy cực kỳ nhẹ
    cov = mne.compute_covariance(epochs_tr, method='ledoit_wolf', verbose=False)
    
    # Fit mô hình xDAWN
    xdawn_model = mne.preprocessing.Xdawn(n_components=n_components, signal_cov=cov)
    xdawn_model.fit(epochs_tr)
    
    del epochs_tr, X_sub, y_sub, cov, events
    gc.collect()
    
    return xdawn_model

def apply_xdawn(xdawn_model, X, sfreq=250):
    """
    Dùng mô hình xDAWN đã fit để transform dữ liệu mới (Train, Val hoặc Test).
    """
    n_trials, n_channels, _ = X.shape
    info = mne.create_info(ch_names=n_channels, sfreq=sfreq, ch_types='eeg')
    
    events = np.column_stack([
        np.arange(n_trials), 
        np.zeros(n_trials, dtype=int), 
        np.ones(n_trials, dtype=int)
    ])
    
    epochs = mne.EpochsArray(X, info, events=events, verbose=False)
    
    X_filt = xdawn_model.transform(epochs) 
    
    return X_filt

def extract_window_amplitude(X, n_windows=5):
    """
    Trích xuất đặc trưng biên độ theo từng cửa sổ thời gian.
    X: (n_trials, n_channels, n_times)
    Returns: Array (n_trials, n_channels * n_windows * 4)
    """
    n_trials, n_channels, n_times = X.shape
    win_len = n_times // n_windows
    features = []

    for w in range(n_windows):
        start = w * win_len
        end = (w + 1) * win_len if w < n_windows - 1 else n_times
        win_data = X[:, :, start:end]

        mean = np.mean(win_data, axis=-1)
        std = np.std(win_data, axis=-1)
        ptp = np.ptp(win_data, axis=-1)  # Biến thiên đỉnh-đối-đỉnh
        rms = np.sqrt(np.mean(win_data**2, axis=-1))  # Giá trị hiệu dụng

        features.extend([mean, std, ptp, rms])

    # Gộp đặc trưng: (n_trials, n_channels, n_windows * 4) -> (n_trials, -1)
    feats = np.stack(features, axis=-1)
    return feats.reshape(n_trials, -1)


def extract_bandpower(X, sfreq=250, bands=None):
    """
    Tính công suất năng lượng theo dải tần sóng EEG.
    X: (n_trials, n_channels, n_times)
    Returns: Array (n_trials, n_channels * n_bands)
    """
    if bands is None:
        bands = {
            'delta': (0.5, 4),
            'theta': (4, 8),
            'alpha': (8, 12),
            'beta': (12, 30),
            'gamma': (30, 45)
        }

    n_trials = X.shape[0]
    nperseg = min(X.shape[-1], int(sfreq))
    freqs, psd = welch(X, fs=sfreq, axis=-1, nperseg=nperseg)

    bp_list = []
    for _, (fmin, fmax) in bands.items():
        idx = np.logical_and(freqs >= fmin, freqs <= fmax)
        # Tích phân Simpson tính công suất dải tần
        bp = simpson(psd[..., idx], x=freqs[idx], axis=-1)
        bp_list.append(bp)

    bp_feats = np.stack(bp_list, axis=-1)  # (n_trials, n_channels, n_bands)
    return bp_feats.reshape(n_trials, -1)

def extract_spectral_entropy(X, sfreq=250):
    """
    Tính Entropy Phổ (Spectral Entropy) cho từng kênh EEG.
    X: (n_trials, n_channels, n_times)
    Returns: Array (n_trials, n_channels)
    """
    n_trials = X.shape[0]
    nperseg = min(X.shape[-1], int(sfreq))
    freqs, psd = welch(X, fs=sfreq, axis=-1, nperseg=nperseg)

    # Chuẩn hóa PSD thành phân bố xác suất
    psd_norm = psd / np.sum(psd, axis=-1, keepdims=True) + 1e-12
    
    # Tính Shannon Entropy trên phổ năng lượng
    spec_entropy = -np.sum(psd_norm * np.log2(psd_norm), axis=-1) / np.log2(psd.shape[-1])
    return spec_entropy.reshape(n_trials, -1)



def extract_wavelet_features(X, wavelet='db4', level=4):
    """
    Phân rã DWT (vector hóa) và trích xuất Năng lượng, Std, Mean của các hệ số.
    X: (n_trials, n_channels, n_times)
    Returns: Array (n_trials, n_channels * (level + 1) * 3)
    """
    n_trials = X.shape[0]

    # Phân rã DWT đồng thời cho toàn bộ trials và channels dọc theo trục thời gian
    coeffs = pywt.wavedec(X, wavelet=wavelet, level=level, axis=-1)

    feats = []
    for coef in coeffs:
        energy = np.sum(coef ** 2, axis=-1)
        std = np.std(coef, axis=-1)
        mean = np.mean(coef, axis=-1)
        feats.extend([energy, std, mean])

    # Ghép thành (n_trials, n_channels, (level + 1) * 3) rồi duỗi thành 2D
    feats = np.stack(feats, axis=-1)
    return feats.reshape(n_trials, -1)

In [6]:
def combine_features(X_raw, X_xdawn, sfreq = 250):
    feat_xdawn = X_xdawn.reshape(X_xdawn.shape[0], -1)
    feat_window = extract_window_amplitude(X_raw)
    feat_band = extract_bandpower(X_raw, sfreq=sfreq)
    feat_se = extract_spectral_entropy(X_raw, sfreq=sfreq)
    feat_wav = extract_wavelet_features(X_raw)

    return np.hstack([feat_xdawn, feat_window, feat_band, feat_se, feat_wav])

def extract_static_features(X_raw, sfreq=250):
    """Tính các đặc trưng KHÔNG phụ thuộc xDAWN (chỉ cần tính 1 lần duy nhất cho mỗi người)"""
    feat_window = extract_window_amplitude(X_raw)
    feat_band = extract_bandpower(X_raw, sfreq=sfreq)
    feat_se = extract_spectral_entropy(X_raw, sfreq=sfreq)
    feat_wav = extract_wavelet_features(X_raw)
    
    return np.hstack([feat_window, feat_band, feat_se, feat_wav]).astype(np.float32)

In [7]:
def precompute_all_data(epochs_dir, sfreq=250):
    all_subjects = [f"{i:02d}" for i in range(1, 43)]
    precomputed = {}
    
    print("Đang tiền tính toán đặc trưng tĩnh cho 42 subjects (Chỉ chạy 1 lần)...")
    for sub_id in all_subjects:
        fif_path = os.path.join(epochs_dir, f"sub-{sub_id}_task-oddball-epo.fif")
        if not os.path.exists(fif_path):
            continue
            
        epochs = mne.read_epochs(fif_path, preload=True, verbose=False)[["standard", "target"]]
        X = epochs.get_data().astype(np.float32)
        y = (epochs.events[:, -1] == 2).astype(np.int8)
        trial_id = epochs.metadata["trial_id"].to_numpy()
        
        # Tính sẵn đặc trưng tĩnh cho từng người (chỉ ~500 trials nên cực nhẹ RAM)
        static_feat = extract_static_features(X, sfreq=sfreq)
        
        precomputed[sub_id] = {
            'X_raw': X,
            'static_feat': static_feat,
            'y': y,
            'trial_id': trial_id
        }
        del epochs; gc.collect()
        print(f"  ✓ Xong Sub-{sub_id} | Static features: {static_feat.shape}")
        
    return precomputed

In [8]:
def train_lda_cross_subject_with_feat(X_base, y_base, X_test_raw, y_test, test_indices, test_sub_id):
    print(f"Train: {X_base.shape[0]} | Test: {X_test_raw.shape[0]}")

    # B1: Fit và áp dụng xDAWN
    xdawn_model = fit_xdawn(X_base, y_base)
    X_base_xdawn = apply_xdawn(xdawn_model, X_base)
    X_test_xdawn = apply_xdawn(xdawn_model, X_test_raw)
    del xdawn_model
    gc.collect()

    # B2: Trích xuất đặc trưng
    X_base_features = combine_features(X_base, X_base_xdawn).astype(np.float32)
    X_test_features = combine_features(X_test_raw, X_test_xdawn).astype(np.float32)
    
    # XÓA NGAY dữ liệu thô và dữ liệu xDAWN vì không còn dùng nữa (Giải phóng ~6GB RAM)
    del X_base, X_base_xdawn, X_test_raw, X_test_xdawn
    gc.collect()

    print(f"Features shape: {X_base_features.shape}")

    # B3: Huấn luyện LDA
    classifier = make_pipeline(
        StandardScaler(),
        LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
    )
    classifier.fit(X_base_features, y_base)
    test_probs = classifier.predict_proba(X_test_features)[:, 1]

    # XÓA NGAY ma trận đặc trưng khổng lồ trước khi thoát hàm
    del X_base_features, X_test_features, classifier
    gc.collect()

    results_df = pd.DataFrame({
        'scheme': 'cross_subject',
        'subject': test_sub_id,
        'fold': 'loso',
        'model': 'LDA-shrink-xDawn',
        'seed': 42,
        'trial_id': test_indices,
        'y_true': y_test,
        'y_score': test_probs,
        'data_version': 'v1', # Thay bằng C.DATA_VERSION nếu có
        'folds_hash': 'cross_sub_all'
    })
    
    return results_df

In [9]:
def train_logreg_cross_subject_with_feat(X_base, y_base, X_test_raw, y_test, test_indices, test_sub_id):
    print(f"Train: {X_base.shape[0]} | Test: {X_test_raw.shape[0]}")

    # B1: Fit và áp dụng xDAWN
    xdawn_model = fit_xdawn(X_base, y_base)
    X_base_xdawn = apply_xdawn(xdawn_model, X_base)
    X_test_xdawn = apply_xdawn(xdawn_model, X_test_raw)
    del xdawn_model
    gc.collect()

    # B2: Trích xuất đặc trưng (ép kiểu float32 để tiết kiệm RAM)
    X_base_features = combine_features(X_base, X_base_xdawn).astype(np.float32)
    X_test_features = combine_features(X_test_raw, X_test_xdawn).astype(np.float32)
    
    # XÓA NGAY dữ liệu thô và dữ liệu xDAWN vì không còn dùng nữa
    del X_base, X_base_xdawn, X_test_raw, X_test_xdawn
    gc.collect()

    print(f"Features shape: {X_base_features.shape}")

    # B3: Huấn luyện Logistic Regression
    classifier = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=42
        )
    )
    classifier.fit(X_base_features, y_base)
    test_probs = classifier.predict_proba(X_test_features)[:, 1]

    # XÓA NGAY ma trận đặc trưng khổng lồ trước khi thoát hàm
    del X_base_features, X_test_features, classifier
    gc.collect()

    results_df = pd.DataFrame({
        'scheme': 'cross_subject',
        'subject': test_sub_id,
        'fold': 'loso',
        'model': 'LogReg-xDAWN',
        'seed': 42,
        'trial_id': test_indices,
        'y_true': y_test,
        'y_score': test_probs,
        'data_version': 'v1', # Thay bằng C.DATA_VERSION nếu có
        'folds_hash': 'cross_sub_all'
    })
    
    return results_df

In [10]:
def train_rf_cross_subject_with_feat(X_base, y_base, X_test_raw, y_test, test_indices, test_sub_id):
    print(f"Train: {X_base.shape[0]} | Test: {X_test_raw.shape[0]}")

    # B1: Fit và áp dụng xDAWN
    xdawn_model = fit_xdawn(X_base, y_base)
    X_base_xdawn = apply_xdawn(xdawn_model, X_base)
    X_test_xdawn = apply_xdawn(xdawn_model, X_test_raw)
    del xdawn_model
    gc.collect()

    # B2: Trích xuất đặc trưng (ép kiểu float32 để tiết kiệm RAM)
    X_base_features = combine_features(X_base, X_base_xdawn).astype(np.float32)
    X_test_features = combine_features(X_test_raw, X_test_xdawn).astype(np.float32)
    
    # XÓA NGAY dữ liệu thô và dữ liệu xDAWN vì không còn dùng nữa
    del X_base, X_base_xdawn, X_test_raw, X_test_xdawn
    gc.collect()

    print(f"Features shape: {X_base_features.shape}")

    # B3: Huấn luyện Random Forest
    classifier = make_pipeline(
        StandardScaler(),
        RandomForestClassifier(
            n_estimators=200,
            class_weight='balanced',
            n_jobs=-1,
            max_depth=10,
            random_state=42
        )
    )
    classifier.fit(X_base_features, y_base)
    test_probs = classifier.predict_proba(X_test_features)[:, 1]

    # XÓA NGAY ma trận đặc trưng khổng lồ trước khi thoát hàm
    del X_base_features, X_test_features, classifier
    gc.collect()

    results_df = pd.DataFrame({
        'scheme': 'cross_subject',
        'subject': test_sub_id,
        'fold': 'loso',
        'model': 'RandomForest-xDAWN',
        'seed': 42,
        'trial_id': test_indices,
        'y_true': y_test,
        'y_score': test_probs,
        'data_version': 'v1', # Thay bằng C.DATA_VERSION nếu có
        'folds_hash': 'cross_sub_all'
    })
    
    return results_df

In [11]:
def train_svm_cross_subject_with_feat(X_base, y_base, X_test_raw, y_test, test_indices, test_sub_id):
    print(f"Train: {X_base.shape[0]} | Test: {X_test_raw.shape[0]}")

    # B1: Fit và áp dụng xDAWN
    xdawn_model = fit_xdawn(X_base, y_base)
    X_base_xdawn = apply_xdawn(xdawn_model, X_base)
    X_test_xdawn = apply_xdawn(xdawn_model, X_test_raw)
    del xdawn_model
    gc.collect()

    # B2: Trích xuất đặc trưng (ép kiểu float32 để tiết kiệm RAM)
    X_base_features = combine_features(X_base, X_base_xdawn).astype(np.float32)
    X_test_features = combine_features(X_test_raw, X_test_xdawn).astype(np.float32)
    
    # XÓA NGAY dữ liệu thô và dữ liệu xDAWN vì không còn dùng nữa
    del X_base, X_base_xdawn, X_test_raw, X_test_xdawn
    gc.collect()

    print(f"Features shape: {X_base_features.shape}")

    # B3: Huấn luyện Support Vector Machine
    classifier = make_pipeline(
        StandardScaler(),
        SVC(
            kernel='linear',          # Dùng 'rbf' (phi tuyến) hoặc 'linear' (tuyến tính)
            probability=True,
            class_weight='balanced',
            random_state=42
        )
    )
    classifier.fit(X_base_features, y_base)
    test_probs = classifier.predict_proba(X_test_features)[:, 1]

    # XÓA NGAY ma trận đặc trưng khổng lồ trước khi thoát hàm
    del X_base_features, X_test_features, classifier
    gc.collect()

    results_df = pd.DataFrame({
        'scheme': 'cross_subject',
        'subject': test_sub_id,
        'fold': 'loso',
        'model': 'SVM-xDAWN',
        'seed': 42,
        'trial_id': test_indices,
        'y_true': y_test,
        'y_score': test_probs,
        'data_version': 'v1', # Thay bằng C.DATA_VERSION nếu có
        'folds_hash': 'cross_sub_all'
    })
    
    return results_df

In [12]:

def get_folds_hash(json_path):
    """Tạo mã hash MD5 trực tiếp từ nội dung file folds.json"""
    with open(json_path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def run_cross_subjects(folds_json_path, epochs_dir, output_path):
    import json
    import gc
    import numpy as np
    import pandas as pd

    with open(folds_json_path, "r", encoding="utf-8") as f:
        raw_json = json.load(f)
    
    # 1. TRUY CẬP ĐÚNG NHÁNH CROSS-SUBJECT
    if "cross_subject" not in raw_json:
        raise ValueError("Không tìm thấy key 'cross_subject' trong file folds.json!")
    
    folds_config = raw_json["cross_subject"]
    folds_hash = get_folds_hash(folds_json_path)
    
    # Load dữ liệu 1 lần vào RAM (chỉ cần dùng X_raw, y, trial_id)
    data_store = precompute_all_data(epochs_dir, sfreq=250)
    valid_subjects = set(data_store.keys())
    all_results = []
    
    for fold_name, fold_info in folds_config.items():
        print(f"\n==========================================")
        print(f"Đang xử lý {fold_name}...")
        
        # 2. BÓC TÁCH SUBJECT CHO TRAIN VÀ TEST
        raw_test_subs = []
        raw_train_subs = []
        
        if isinstance(fold_info, list):
            raw_test_subs = fold_info
        elif isinstance(fold_info, dict):
            for k, v in fold_info.items():
                if 'test' in k.lower():
                    raw_test_subs = v
                elif 'train' in k.lower():
                    raw_train_subs = v
                    
        if not raw_test_subs:
            print(f"⚠️ Bỏ qua {fold_name}: Không tìm thấy danh sách test subjects.")
            continue
            
        # Chuẩn hóa tên subject thành chuỗi '01', '02'...
        test_subs = [str(s).replace("sub-", "").strip().zfill(2) for s in raw_test_subs]
        test_subs = [s for s in test_subs if s in valid_subjects]
        
        if raw_train_subs:
            train_subs = [str(s).replace("sub-", "").strip().zfill(2) for s in raw_train_subs]
            train_subs = [s for s in train_subs if s in valid_subjects]
        else:
            train_subs = [s for s in valid_subjects if s not in test_subs]
            
        if len(train_subs) == 0 or len(test_subs) == 0:
            print(f"⚠️ Bỏ qua {fold_name}: Tập Train ({len(train_subs)}) hoặc Test ({len(test_subs)}) bị trống!")
            continue
            
        print(f"Train subjects ({len(train_subs)}): {sorted(train_subs)[:5]}...")
        print(f"Test subjects  ({len(test_subs)}): {sorted(test_subs)}")
        
        # 3. GOM DỮ LIỆU TRAIN / TEST
        X_tr_raw = np.concatenate([data_store[s]['X_raw'] for s in train_subs], axis=0)
        y_tr = np.concatenate([data_store[s]['y'] for s in train_subs], axis=0)
        
        X_te_raw = np.concatenate([data_store[s]['X_raw'] for s in test_subs], axis=0)
        y_te = np.concatenate([data_store[s]['y'] for s in test_subs], axis=0)
        te_indices = np.concatenate([data_store[s]['trial_id'] for s in test_subs], axis=0)
        te_sub_ids = np.concatenate([np.full(len(data_store[s]['y']), s) for s in test_subs], axis=0)
        
        # 4. GỌI HÀM TRAIN SVM (chỉ fit 1 lần cho toàn bộ test subjects của fold)
        print(f"--- {fold_name} ---")
        df_fold = train_svm_cross_subject_with_feat(
            X_base=X_tr_raw,
            y_base=y_tr,
            X_test_raw=X_te_raw,
            y_test=y_te,
            test_indices=te_indices,
            test_sub_id=te_sub_ids
        )
        
        # 5. GHI ĐÈ METADATA CHO ĐÚNG KỊCH BẢN CROSS-SUBJECT
        df_fold['scheme'] = 'cross_subject'
        df_fold['fold'] = fold_name
        df_fold['folds_hash'] = folds_hash
        
        all_results.append(df_fold)
        del X_tr_raw, X_te_raw, y_tr, y_te, te_indices, te_sub_ids; gc.collect()
        
        # Lưu checkpoint sau mỗi fold
        pd.concat(all_results, ignore_index=True).to_csv(output_path, index=False)
        print(f"✅ Hoàn thành {fold_name}!")

    print(f"\n🎉 HOÀN TẤT 5-FOLD CROSS-SUBJECT (SVM)! File lưu tại: {output_path}")

In [13]:
def run_within_subjects(folds_json_path, epochs_dir, output_path):
    import json
    import gc
    import numpy as np
    import pandas as pd

    with open(folds_json_path, "r", encoding="utf-8") as f:
        raw_json = json.load(f)
    
    # 1. TRUY CẬP ĐÚNG NHÁNH WITHIN-SUBJECT
    if "within_subject" not in raw_json:
        raise ValueError("Không tìm thấy key 'within_subject' trong file folds.json!")
    
    within_config = raw_json["within_subject"]
    folds_hash = get_folds_hash(folds_json_path)
    
    # Load dữ liệu 1 lần vào RAM (chỉ cần dùng X_raw, y, trial_id)
    data_store = precompute_all_data(epochs_dir, sfreq=250)
    valid_subjects = set(data_store.keys())
    all_results = []
    
    for raw_sub_id, folds_dict in within_config.items():
        # Chuẩn hóa tên subject thành chuỗi '01', '02'...
        sub_id = str(raw_sub_id).replace("sub-", "").strip().zfill(2)
        if sub_id not in valid_subjects:
            print(f"⚠️ Bỏ qua Subject {sub_id}: Không có dữ liệu trong data_store.")
            continue
            
        print(f"\n==========================================")
        print(f"Đang xử lý Subject {sub_id} (LogReg)...")
        
        X_raw_all = data_store[sub_id]['X_raw']
        y_all = data_store[sub_id]['y']
        trial_ids_all = np.array(data_store[sub_id]['trial_id'])
        
        for fold_name, fold_info in folds_dict.items():
            # 2. BÓC TÁCH TRIAL_ID CHO TRAIN VÀ TEST
            raw_test_ids = []
            raw_train_ids = []
            
            if isinstance(fold_info, list):
                raw_test_ids = fold_info
            elif isinstance(fold_info, dict):
                for k, v in fold_info.items():
                    if 'test' in k.lower():
                        raw_test_ids = v
                    elif 'train' in k.lower():
                        raw_train_ids = v
                        
            if not raw_test_ids:
                print(f"⚠️ Bỏ qua {sub_id} | {fold_name}: Không tìm thấy danh sách test trial_id.")
                continue
                
            test_mask = np.isin(trial_ids_all, raw_test_ids)
            train_mask = np.isin(trial_ids_all, raw_train_ids) if raw_train_ids else ~test_mask
                
            if np.sum(train_mask) == 0 or np.sum(test_mask) == 0:
                print(f"⚠️ Bỏ qua {sub_id} | {fold_name}: Tập Train hoặc Test bị trống!")
                continue
                
            # 3. TÁCH DỮ LIỆU TRAIN / TEST
            X_tr_raw, X_te_raw = X_raw_all[train_mask], X_raw_all[test_mask]
            y_tr, y_te = y_all[train_mask], y_all[test_mask]
            te_indices = trial_ids_all[test_mask]
            
            # 4. GỌI HÀM TRAIN LOGREG
            print(f"--- Subject {sub_id} | {fold_name} ---")
            df_fold = train_svm_cross_subject_with_feat(
                X_base=X_tr_raw,
                y_base=y_tr,
                X_test_raw=X_te_raw,
                y_test=y_te,
                test_indices=te_indices,
                test_sub_id=sub_id
            )
            
            # 5. GHI ĐÈ METADATA CHO ĐÚNG KỊCH BẢN WITHIN-SUBJECT
            df_fold['scheme'] = 'within_subject'
            df_fold['fold'] = fold_name
            df_fold['folds_hash'] = folds_hash
            
            all_results.append(df_fold)
            del X_tr_raw, X_te_raw, y_tr, y_te; gc.collect()
            
        # Lưu checkpoint sau mỗi subject
        pd.concat(all_results, ignore_index=True).to_csv(output_path, index=False)
        print(f"✅ Hoàn thành Subject {sub_id}!")

    print(f"\n🎉 HOÀN TẤT 5-FOLD WITHIN-SUBJECT (LogReg)! File lưu tại: {output_path}")

In [14]:
run_cross_subjects(FOLDS_JSON_PATH, EPOCHS_DIR, OUTPUT_CSV)

Đang tiền tính toán đặc trưng tĩnh cho 42 subjects (Chỉ chạy 1 lần)...
  ✓ Xong Sub-01 | Static features: (538, 5207)
  ✓ Xong Sub-02 | Static features: (527, 5207)
  ✓ Xong Sub-03 | Static features: (547, 5207)
  ✓ Xong Sub-04 | Static features: (575, 5207)
  ✓ Xong Sub-05 | Static features: (551, 5207)
  ✓ Xong Sub-06 | Static features: (553, 5207)
  ✓ Xong Sub-07 | Static features: (303, 5207)
  ✓ Xong Sub-08 | Static features: (543, 5207)
  ✓ Xong Sub-09 | Static features: (563, 5207)
  ✓ Xong Sub-10 | Static features: (540, 5207)
  ✓ Xong Sub-11 | Static features: (568, 5207)
  ✓ Xong Sub-12 | Static features: (575, 5207)
  ✓ Xong Sub-13 | Static features: (575, 5207)
  ✓ Xong Sub-14 | Static features: (445, 5207)
  ✓ Xong Sub-15 | Static features: (573, 5207)
  ✓ Xong Sub-16 | Static features: (571, 5207)
  ✓ Xong Sub-17 | Static features: (506, 5207)
  ✓ Xong Sub-18 | Static features: (481, 5207)
  ✓ Xong Sub-19 | Static features: (570, 5207)
  ✓ Xong Sub-20 | Static features: (

/tmp/ipykernel_16/472685025.py:39: RuntimeWarning: Epochs are not baseline corrected, covariance matrix may be inaccurate
  cov = mne.compute_covariance(epochs_tr, method='ledoit_wolf', verbose=False)


Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Features shape: (18073, 6211)
✅ Hoàn thành fold_0!

Đang xử lý fold_1...
Train subjects (33): ['01', '02', '03', '04', '05']...
Test subjects  (9): ['07', '10', '13', '16', '17', '18', '28', '34', '37']
--- fold_1 ---
Train: 18358 | Test: 4623


/tmp/ipykernel_16/472685025.py:39: RuntimeWarning: Epochs are not baseline corrected, covariance matrix may be inaccurate
  cov = mne.compute_covariance(epochs_tr, method='ledoit_wolf', verbose=False)


Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Features shape: (18358, 6211)
✅ Hoàn thành fold_1!

Đang xử lý fold_2...
Train subjects (34): ['03', '04', '05', '07', '08']...
Test subjects  (8): ['01', '02', '06', '12', '25', '32', '36', '41']
--- fold_2 ---
Train: 18652 | Test: 4329


/tmp/ipykernel_16/472685025.py:39: RuntimeWarning: Epochs are not baseline corrected, covariance matrix may be inaccurate
  cov = mne.compute_covariance(epochs_tr, method='ledoit_wolf', verbose=False)


Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Features shape: (18652, 6211)
✅ Hoàn thành fold_2!

Đang xử lý fold_3...
Train subjects (34): ['01', '02', '05', '06', '07']...
Test subjects  (8): ['03', '04', '22', '24', '33', '35', '38', '42']
--- fold_3 ---
Train: 18355 | Test: 4626


/tmp/ipykernel_16/472685025.py:39: RuntimeWarning: Epochs are not baseline corrected, covariance matrix may be inaccurate
  cov = mne.compute_covariance(epochs_tr, method='ledoit_wolf', verbose=False)


Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Features shape: (18355, 6211)
✅ Hoàn thành fold_3!

Đang xử lý fold_4...
Train subjects (34): ['01', '02', '03', '04', '05']...
Test subjects  (8): ['08', '11', '15', '19', '21', '23', '29', '39']
--- fold_4 ---
Train: 18486 | Test: 4495


/tmp/ipykernel_16/472685025.py:39: RuntimeWarning: Epochs are not baseline corrected, covariance matrix may be inaccurate
  cov = mne.compute_covariance(epochs_tr, method='ledoit_wolf', verbose=False)


Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Features shape: (18486, 6211)
✅ Hoàn thành fold_4!

🎉 HOÀN TẤT 5-FOLD CROSS-SUBJECT (SVM)! File lưu tại: /kaggle/working/final_predictions_cross_subject.csv
